# Measuring a Classifier

**CS 212 AI Programming 1**

This is the Jupyter notebook version of the [Lab 4 example](https://lcc-cit.github.io/CS212-CourseMaterials/Examples/Lab04-ClassifierMetrics/CS212_Lab04_Example.html). It builds on the spam detection example (see the [Spam Detection overview](https://lcc-cit.github.io/CS212-CourseMaterials/Examples/SpamDetection/CS212_SpamDetection_Overview.html)). It takes the spam classifier you already know and shows what its performance metrics mean and how they are calculated.

*Accuracy*, *precision*, *recall* and *F1* are all **performance metrics**: numbers that measure how well a classifier does its job. By the end of this notebook you will know how each one is calculated, and how to get it from scikit-learn.

## 1. Starting Point: The Spam Classifier

The code cell below repeats the steps of the Spam Detection example. It loads the SMS Spam Collection (5,574 real text messages that people labeled as spam or ham), holds back 25% of the messages for testing, counts the words, trains a naive Bayes classifier, and classifies the test messages. We won't explain those steps again. The Spam Detection notebook explains each one.

There are two small differences. We read the data with Python's `csv` module, and we use numbers for the labels: **1 means spam and 0 means ham**.

In [1]:
import csv
import io
from urllib.request import urlopen

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

DATA_URL = "https://raw.githubusercontent.com/LCC-CIT/CS212-CourseMaterials/main/docs/Examples/SpamDetection/SpamDetectionNotebook/sms_spam.csv"

# Read the data. Each row has a label (spam or ham) and the text of one message.
messages = []
labels = []
with urlopen(DATA_URL) as response:
    text = response.read().decode("utf-8")
for row in csv.DictReader(io.StringIO(text)):
    messages.append(row["message"])
    if row["label"] == "spam":
        labels.append(1)
    else:
        labels.append(0)

# The same split, word counts and classifier as the Spam Detection example.
train_messages, test_messages, train_labels, test_labels = train_test_split(
    messages, labels, test_size=0.25, random_state=42)

vectorizer = CountVectorizer()
train_counts = vectorizer.fit_transform(train_messages)
test_counts = vectorizer.transform(test_messages)

classifier = MultinomialNB()
classifier.fit(train_counts, train_labels)
predicted_labels = classifier.predict(test_counts).tolist()   # a plain list of 0s and 1s

print(len(messages), "messages,", sum(labels), "of them are spam")
print(len(test_messages), "test messages,", sum(test_labels), "of them are spam")

5574 messages, 747 of them are spam
1394 test messages, 191 of them are spam


## 2. What We Are Measuring

To find out how well a classifier works, you compare its answers with the right answers for the same messages. Both are needed:

- The **true labels** are the correct classes. People decided them: someone read each message and marked it spam or ham.
- The **predicted labels** are the classes the classifier chose for the same messages.

We measure the classifier on the *test* messages, which it did not learn from. Here are the first five test messages that really are spam, with the label that the classifier predicted for each:

In [2]:
shown = 0
for i in range(len(test_messages)):
    if test_labels[i] == 1 and shown < 5:
        print("true:", test_labels[i], "| predicted:", predicted_labels[i], "|", test_messages[i][:60])
        shown += 1

true: 1 | predicted: 1 | FREE MESSAGE Activate your 500 FREE Text Messages by replyin
true: 1 | predicted: 1 | T-Mobile customer you may now claim your FREE CAMERA PHONE u
true: 1 | predicted: 1 | Kit Strip - you have been billed 150p. Netcollex Ltd. PO Box
true: 1 | predicted: 1 | URGENT This is our 2nd attempt to contact U. Your £900 prize
true: 1 | predicted: 1 | Call Germany for only 1 pence per minute! Call from a fixed 


## 3. Why Accuracy Isn't Enough

*Accuracy* is the percent of the messages that the classifier got right. `accuracy_score` in scikit-learn calculates it. We'll compare the spam classifier with a "classifier" that never needs any training: it calls every message ham.

In [3]:
from sklearn.metrics import accuracy_score

always_ham = [0] * len(test_labels)

print("Spam classifier:", round(accuracy_score(test_labels, predicted_labels), 3))
print("Always ham:     ", round(accuracy_score(test_labels, always_ham), 3))

Spam classifier: 0.986
Always ham:      0.863


The spam classifier's accuracy of 98.6% sounds excellent, but the "always ham" classifier is already 86.3% accurate, because most messages are ham. It also never finds a single spam message, which makes it useless as a spam filter. And if spam were much rarer than 13%, the problem would be worse: a classifier that is 99% accurate when 99% of the messages are ham is no better than "always ham".

To see how well a classifier does on the rare class, we need to count the different kinds of right and wrong answers.

## 4. The Four Counts

Pick one class to be the *positive* class. Here it is spam (label 1). Every prediction is one of four kinds:

| | Predicted spam | Predicted ham |
| --- | --- | --- |
| **Really spam** | True positive (TP) | False negative (FN) |
| **Really ham** | False positive (FP) | True negative (TN) |

A table like this is called a *confusion matrix*. scikit-learn's `confusion_matrix` function counts the four kinds for you. For two classes, its result is `[[TN, FP], [FN, TP]]`, so `.ravel()` flattens it into the four numbers in that order.

In [4]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(test_labels, predicted_labels, labels=[0, 1]).ravel()
print("True positives: ", tp, "(spam that it called spam)")
print("False positives:", fp, "(ham that it called spam)")
print("False negatives:", fn, "(spam that it called ham)")
print("True negatives: ", tn, "(ham that it called ham)")

True positives:  178 (spam that it called spam)
False positives: 6 (ham that it called spam)
False negatives: 13 (spam that it called ham)
True negatives:  1197 (ham that it called ham)


## 5. Calculating the Performance Metrics

From the four counts you can calculate these performance metrics:

| Measure | Formula | The question it answers |
| --- | --- | --- |
| Accuracy (percent correct) | (TP + TN) / all messages | What percent of all the messages did it get right? |
| Precision | TP / (TP + FP) | When it says "spam", how often is it right? |
| Recall | TP / (TP + FN) | Of the real spam messages, how many did it find? |
| F1 | 2 × precision × recall / (precision + recall) | One number that is high only when precision *and* recall are both high |

You don't need to write code to calculate them. `sklearn.metrics` has a function for each one: `accuracy_score`, `precision_score`, `recall_score` and `f1_score`. You give each of them the true labels and the predicted labels. If a classifier never says "spam", then TP + FP is 0, and precision would divide by zero. The argument `zero_division=0` tells `precision_score` to return 0.0 in that case instead of printing a warning.

The function below prints the four counts and the four performance metrics. We use it for the spam classifier and for the "always ham" classifier.

In [5]:
from sklearn.metrics import f1_score, precision_score, recall_score


def show_scores(name, true_labels, predicted_labels):
    tn, fp, fn, tp = confusion_matrix(true_labels, predicted_labels, labels=[0, 1]).ravel()
    print(name)
    print("  counts:  TP =", tp, " FP =", fp, " FN =", fn, " TN =", tn)
    print("  accuracy ", round(accuracy_score(true_labels, predicted_labels), 3))
    print("  precision", round(precision_score(true_labels, predicted_labels, zero_division=0), 3))
    print("  recall   ", round(recall_score(true_labels, predicted_labels), 3))
    print("  F1       ", round(f1_score(true_labels, predicted_labels), 3))


show_scores("Spam classifier", test_labels, predicted_labels)
print()
show_scores("Always ham", test_labels, always_ham)

Spam classifier
  counts:  TP = 178  FP = 6  FN = 13  TN = 1197
  accuracy  0.986
  precision 0.967
  recall    0.932
  F1        0.949

Always ham
  counts:  TP = 0  FP = 0  FN = 191  TN = 1203
  accuracy  0.863
  precision 0.0
  recall    0.0
  F1        0.0


## 6. Reading the Numbers

**The "always ham" classifier.** Its accuracy is 86.3%, but its recall is 0.0: it found none of the 191 spam messages. Its precision is also reported as 0.0, but really it is undefined, because the classifier never said "spam". When one class is rare, accuracy mostly measures how common the *other* class is, so it can't tell a useful classifier from a useless one. Precision and recall look at the rare class itself.

**The spam classifier.** Its accuracy of 98.6% is far above the 86.3% of "always ham", so the classifier really is useful. Precision and recall show what kind of mistakes it makes. It said "spam" for 184 of the 1,394 test messages (178 + 6). Its precision is 0.967 because 178 of those 184 really were spam, and the other 6 were normal messages that it wrongly called spam. Its recall is 0.932 because there were 191 spam messages in the test set, and it found 178 of them. The other 13 got through.

**Which mistake is worse?** That depends on what the classifier is for. For a spam filter, losing a real message (a false positive) is probably worse than letting some spam through (a false negative), so precision matters more. For a classifier that flags emergency messages, it would be the other way around, and recall would matter more. You choose the performance metrics to watch by thinking about what each kind of mistake costs.

## 7. The Classification Report and the Confusion Matrix

The Spam Detection example printed a classification report and a confusion matrix. Now you can read them. The report has the precision, recall and F1 of *each* class, treating that class as the positive one. The row for `spam` has the same numbers that `show_scores` printed. In the confusion matrix, each row is a true class and each column is a predicted class, so the numbers on the diagonal are the correct answers.

In [6]:
from sklearn.metrics import classification_report

print(classification_report(test_labels, predicted_labels, target_names=["ham", "spam"]))
print("Confusion matrix (rows = true, columns = predicted):")
print(["ham", "spam"])
print(confusion_matrix(test_labels, predicted_labels))

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99      1203
        spam       0.97      0.93      0.95       191

    accuracy                           0.99      1394
   macro avg       0.98      0.96      0.97      1394
weighted avg       0.99      0.99      0.99      1394

Confusion matrix (rows = true, columns = predicted):
['ham', 'spam']
[[1197    6]
 [  13  178]]


The row for really ham is `[1197 6]` and the row for really spam is `[13 178]`. Those are the same four counts as before.

## 8. More Than Two Classes

The spam classifier chooses between two classes. Of the three lab assignment versions, Group B's classifier chooses between six classes (six departments), so if you are in Group B, you will read a report like the one below. When there are more than two classes, scikit-learn calculates precision, recall and F1 for each class in turn, treating that class as the positive one. To show what that looks like, here is the newsgroups example from the scikit-learn text tutorial in Week 3: posts from four newsgroups (computer graphics, hockey, medicine and religion), classified with the tf-idf and linear SVM pipeline from the tutorial. Two of the tutorial's newsgroups (atheism and Christianity) are replaced with hockey and religion. To show what happens when one class is rare, the function `keep_fewer` throws away most of the religion posts: it keeps only 75 of the 377 for training and 50 of the 251 for testing. The other three newsgroups keep all of their posts, about 600 each for training and 390 each for testing. The first run downloads the data, so it needs an internet connection.

In [7]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline


def keep_fewer(newsgroups, small_group, how_many):
    """Return the posts and their labels, keeping only the first how_many posts of small_group."""
    small_label = newsgroups.target_names.index(small_group)
    posts = []
    labels = []
    small_count = 0
    for i in range(len(newsgroups.data)):
        label = newsgroups.target[i]
        if label == small_label:
            small_count = small_count + 1
        if label != small_label or small_count <= how_many:
            posts.append(newsgroups.data[i])
            labels.append(label)
    return posts, labels


categories = ["talk.religion.misc", "rec.sport.hockey", "comp.graphics", "sci.med"]
twenty_train = fetch_20newsgroups(subset="train", categories=categories, shuffle=True, random_state=42)
twenty_test = fetch_20newsgroups(subset="test", categories=categories, shuffle=True, random_state=42)

# Keep only 75 of the 377 religion posts for training and 50 of the 251 for testing.
train_posts, train_groups = keep_fewer(twenty_train, "talk.religion.misc", 75)
test_posts, test_groups = keep_fewer(twenty_test, "talk.religion.misc", 50)

text_clf = Pipeline([
    ("vect", CountVectorizer()),
    ("tfidf", TfidfTransformer()),
    ("clf", SGDClassifier(loss="hinge", penalty="l2", alpha=1e-3,
                          random_state=42, max_iter=5, tol=None)),
])
text_clf.fit(train_posts, train_groups)
newsgroup_predictions = text_clf.predict(test_posts)

print(classification_report(test_groups, newsgroup_predictions,
                            target_names=twenty_test.target_names))
print("Confusion matrix (rows = true, columns = predicted):")
print(twenty_test.target_names)
print(confusion_matrix(test_groups, newsgroup_predictions))

                    precision    recall  f1-score   support

     comp.graphics       0.90      0.97      0.93       389
  rec.sport.hockey       0.97      0.99      0.98       399
           sci.med       0.93      0.90      0.92       396
talk.religion.misc       0.96      0.52      0.68        50

          accuracy                           0.94      1234
         macro avg       0.94      0.84      0.88      1234
      weighted avg       0.94      0.94      0.93      1234

Confusion matrix (rows = true, columns = predicted):
['comp.graphics', 'rec.sport.hockey', 'sci.med', 'talk.religion.misc']
[[377   3   9   0]
 [  3 394   2   0]
 [ 35   3 357   1]
 [  4   6  14  26]]


Read the report one row at a time. For `talk.religion.misc`, the recall is 0.52: there were 50 religion posts in the test set and the classifier found only 26 of them. The precision is 0.96: the classifier put 27 posts in `talk.religion.misc` (0 + 0 + 1 + 26, the last column of the confusion matrix) and 26 of them belonged there. The other classes work the same way, each one taking its turn as the positive class. In the confusion matrix each row is a true newsgroup and each column is a predicted newsgroup, so the numbers on the diagonal are the correct answers.

At the bottom of the report are two kinds of average:

- **Macro average:** the plain average of the per-class scores. Every class counts the same, however small.
- **Weighted average:** the per-class scores averaged in proportion to the number of posts in each class (the *support* column).

Here the religion class is much smaller than the others: 50 test posts, compared with 389 to 399. In the weighted average it counts for only 50 of the 1234 posts, so the weighted average F1 is 0.93, close to the scores of the three big classes. In the macro average it counts as one class out of four, so its low recall pulls the macro average recall down to 0.84 and the macro average F1 down to 0.88. When the classes are not the same size, look at both averages and at each class's own row.

The accuracy of 0.94 hides the fact that the classifier misses about half of the religion posts. It is the same problem as the "always ham" classifier: a rare class hardly changes the accuracy, so the accuracy can't tell you how well that class is classified. The confusion matrix shows where the missed posts went: 14 of the 24 were put in `sci.med`. With only 75 religion posts to learn from, the classifier rarely predicts that class. When it does, it is almost always right (high precision), but it misses many of the religion posts (low recall). The best fix is to collect more examples of the rare class. When you can't, you can at least choose a classifier's settings with a score that counts every class equally, as the next section shows.

## 9. Using a Metric to Tune a Classifier

Performance metrics are also how you decide whether a change made a classifier better. Most classifiers have *hyperparameters*, settings that you choose before training, such as the `alpha` of naive Bayes or whether the word counts include pairs of words. `GridSearchCV` tries every combination of the settings you give it. For each combination it trains and tests the classifier several times with cross-validation, using only the training messages. Then it keeps the combination with the best *score*.

That score is another performance metric, and you choose it with the `scoring` argument. The default for a classifier is accuracy, which has the problem described above: when one class is rare, accuracy can hardly tell a good classifier from a bad one. For a rare class you usually want `scoring="f1"`, which is the F1 of the positive class. For more than two classes, `scoring="f1_macro"` averages the F1 of every class, the same way the macro average in the report does.

The code below puts the word counts and the classifier in a `Pipeline`, so the grid search can change settings in both, and tries a small grid with each of the two scores. The names `"counts"` and `"nb"` are used in the settings: `nb__alpha` is the `alpha` setting of the step named `nb`.

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

pipeline = Pipeline([("counts", CountVectorizer()), ("nb", MultinomialNB())])
settings = {
    "counts__ngram_range": [(1, 1), (1, 2)],   # single words, or single words and word pairs
    "nb__alpha": [0.1, 0.5, 1.0],              # how much smoothing naive Bayes does
}

# Cross-validation on the training messages only. We try two scores: accuracy and
# F1 (the F1 of the positive class, spam).
for score_name in ["accuracy", "f1"]:
    search = GridSearchCV(pipeline, settings, scoring=score_name, cv=5)
    search.fit(train_messages, train_labels)
    print("Best settings when scoring by", score_name)
    print("  ", search.best_params_)
    print("   cross-validation", score_name, "=", round(search.best_score_, 3))

# Only now use the test messages, once, with the model that was best by F1.
print()
tuned_labels = search.predict(test_messages)
print(classification_report(test_labels, tuned_labels, target_names=["ham", "spam"]))

Best settings when scoring by accuracy
   {'counts__ngram_range': (1, 2), 'nb__alpha': 0.1}
   cross-validation accuracy = 0.987
Best settings when scoring by f1
   {'counts__ngram_range': (1, 2), 'nb__alpha': 0.1}
   cross-validation f1 = 0.95

              precision    recall  f1-score   support

         ham       0.99      0.99      0.99      1203
        spam       0.96      0.94      0.95       191

    accuracy                           0.99      1394
   macro avg       0.97      0.97      0.97      1394
weighted avg       0.99      0.99      0.99      1394



Both scores picked the same settings, and the tuned classifier is barely different from the untuned one: spam precision went from 0.967 to 0.957 and recall from 0.932 to 0.942, so F1 stayed at about 0.95. That is a normal result. The spam classifier was already good, the test set is small (191 spam messages), and a difference of one or two messages is just noise. Tuning isn't a way to get a big improvement. It helps you pick settings in a systematic way, and it only makes sense if the score you tune for is the one that matters. Here the two scores agree because spam is 13% of the messages. The rarer the class is, the more likely they are to disagree.

## 10. Tuning the Newsgroups Classifier

The newsgroups classifier from section 8 is a better test, because its religion class is rare. The grid below has one more setting, `class_weight`. Normally every training post counts the same, so a class with only 75 posts has little effect on what the classifier learns. The setting `class_weight="balanced"` makes each post of a small class count for more, in proportion to how rare the class is. This time the two scores are accuracy and `f1_macro`, the average of the F1 of each newsgroup. To see how the two choices differ, both tuned classifiers are tested on the test posts. The cell takes about a minute to run.

In [9]:
pipeline = Pipeline([
    ("vect", CountVectorizer()),
    ("tfidf", TfidfTransformer()),
    ("clf", SGDClassifier(loss="hinge", penalty="l2", random_state=42, max_iter=5, tol=None)),
])
settings = {
    "vect__ngram_range": [(1, 1), (1, 2)],          # single words, or single words and word pairs
    "clf__alpha": [0.01, 0.001, 0.0001, 0.00001],   # how strongly the classifier is kept simple
    "clf__class_weight": [None, "balanced"],        # "balanced" counts each post of a small class for more
}

for score_name in ["accuracy", "f1_macro"]:
    search = GridSearchCV(pipeline, settings, scoring=score_name, cv=5)
    search.fit(train_posts, train_groups)
    print("Best settings when scoring by", score_name)
    print("  ", search.best_params_)
    print("   cross-validation", score_name, "=", round(search.best_score_, 3))
    print()
    predicted_groups = search.predict(test_posts)
    print(classification_report(test_groups, predicted_groups,
                                target_names=twenty_test.target_names))

Best settings when scoring by accuracy
   {'clf__alpha': 0.0001, 'clf__class_weight': 'balanced', 'vect__ngram_range': (1, 2)}
   cross-validation accuracy = 0.976

                    precision    recall  f1-score   support

     comp.graphics       0.91      0.97      0.94       389
  rec.sport.hockey       0.97      0.99      0.98       399
           sci.med       0.96      0.90      0.93       396
talk.religion.misc       0.82      0.72      0.77        50

          accuracy                           0.94      1234
         macro avg       0.91      0.89      0.90      1234
      weighted avg       0.94      0.94      0.94      1234

Best settings when scoring by f1_macro
   {'clf__alpha': 0.0001, 'clf__class_weight': 'balanced', 'vect__ngram_range': (1, 1)}
   cross-validation f1_macro = 0.966

                    precision    recall  f1-score   support

     comp.graphics       0.91      0.97      0.94       389
  rec.sport.hockey       0.98      0.99      0.98       399
      

Tuning made a much bigger difference here than it did for spam. Before tuning, in section 8, the classifier found 26 of the 50 religion posts (recall 0.52). Both tuned classifiers use a smaller `alpha` and `class_weight="balanced"`, and they find many more: 36 with the settings chosen by accuracy (recall 0.72) and 42 with the settings chosen by `f1_macro` (recall 0.84). The two scores disagree about word pairs. The F1 of the religion class went from 0.68 to 0.77 and 0.83, and the macro average F1 from 0.88 to 0.90 and 0.92. There is a cost: the precision of the religion class went down from 0.96 to 0.82, because the classifier now puts more posts in that class that don't belong there.

Now look at the accuracy on the test posts: it is 0.94 for both tuned classifiers, the same as before tuning. If accuracy were the only number you looked at, you would think tuning had done nothing. Keep in mind that there are only 50 religion posts in the test set, so the difference between the two tuned classifiers is 6 posts. It is a real difference but not a big one. The lesson is that the score you tune for decides which settings you get, so it should be the score that measures what you care about.

Notice that the test messages and posts were not used to choose the settings. The grid search chose them with cross-validation on the training data, and the test data was used only at the end, to see how well the tuned classifiers work. (For the newsgroups, both tuned classifiers are tested so that you can compare them.) If you chose the settings by looking at the test results, the test results would no longer tell you how well the classifier works on new messages.

## Try It Yourself

- Make a classifier that calls every message spam: `always_spam = [1] * len(test_labels)`. Use `show_scores` on it. What are its precision and recall? Why is it useless even though its recall is perfect?
- In the first code cell, change `test_size` from 0.25 to 0.5 and run the notebook again. Which of the numbers change, and by how much? What does that tell you about how much to trust the result from one test set?
- Which mistake would you rather your spam filter make: sending a real message to the spam folder, or letting a spam message through? What do the precision and recall of spam tell you about that?
- In the grid search cell, add `0.01` to the `nb__alpha` list and `(1, 3)` to the `counts__ngram_range` list. Do the best settings change? Does the F1 on the test messages change? If not, what does that tell you about how much there was to gain from tuning this classifier?
- In the newsgroups example, look at the confusion matrix. Which two newsgroups are confused most often? Calculate the precision and recall of `sci.med` from the matrix, and check your answers against the report.
- In the newsgroups code cell, change the 75 and the 50 to 377 and 251, so that all of the religion posts are used. How do the recall of `talk.religion.misc` and the macro average change? Run the newsgroups grid search cell again: does tuning still make as much difference? Try 150 and 100 as well.

*This notebook was drafted by Claude Code and reviewed by Brian Bird. 10/21/2026*